# 11 — HITL feedback round-trip

Demonstrates the full loop: pipeline run → human feedback → feedback applied →
measurably different artifacts. Thin orchestration only; semantics live in
`src/reviewscope_ml/hitl/apply_feedback.py` (its module docstring is the
authoritative description of what each action does on re-run).

Normally feedback comes from the Streamlit app
(`streamlit run src/reviewscope_ml/hitl/app.py`); here we write the same
records programmatically so the notebook runs unattended.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from reviewscope_ml import load_config
from reviewscope_ml.data import load_benchmark
from reviewscope_ml.pipelines import load_run

cfg = load_config(sample_size=1_000, device='cpu')
reviews = load_benchmark(cfg)

# Two-stage run from notebook 10 — its micro→macro hierarchy lets a split
# be answered without re-clustering.
RUN = f'two_stage__{cfg.sample_size}__s42'
art = load_run(cfg.runs_dir / RUN)
print(f'{RUN}: {len(art.clusters)} clusters')

In [ ]:
# Before: cluster overview
def overview(a):
    return pd.DataFrame([
        {'cluster': cid, 'label': i.label, 'size': i.size,
         'source': i.label_source,
         'micro_ids': i.micro_cluster_ids,
         'top_terms': ', '.join(w for w, _ in (tuple(t) for t in i.top_terms[:5]))}
        for cid, i in sorted(a.clusters.items())
    ])

before = overview(art)
before

In [ ]:
# A reviewer session, written as records (the Streamlit app produces exactly these)
from reviewscope_ml.hitl import FeedbackRecord, append_record, session_file

cids = art.cluster_ids
session = session_file(cfg.feedback_dir, RUN)
actions = [
    FeedbackRecord(RUN, 'demo-reviewer', 'rename_label', cluster_id=cids[0],
                   new_label='Front desk & check-in experience'),
    FeedbackRecord(RUN, 'demo-reviewer', 'approve_label', cluster_id=cids[1]),
    FeedbackRecord(RUN, 'demo-reviewer', 'merge_clusters', cluster_id=cids[3],
                   merge_into=cids[2], note='same theme, split by phrasing'),
    FeedbackRecord(RUN, 'demo-reviewer', 'split_cluster', cluster_id=cids[4],
                   note='mixes two distinct complaints'),
]
for r in actions:
    append_record(session, r)
print(f'{len(actions)} records -> {session.name}')

In [ ]:
# Apply: produces <run>__reviewed next to the original (original untouched)
from reviewscope_ml.hitl import apply_run_feedback

reviewed_dir = apply_run_feedback(cfg.runs_dir / RUN, cfg.feedback_dir, reviews=reviews)
reviewed = load_run(reviewed_dir)
after = overview(reviewed)
after

In [ ]:
# What changed?
print('applied:', *reviewed.manifest['feedback_applied'], sep='\n  - ')
print()
print(f'clusters before: {len(art.clusters)}  after: {len(reviewed.clusters)}')
print('needs_recluster:', reviewed.manifest.get('needs_recluster'))

## What each demonstrated action did

- **rename_label** — label overridden, `label_source='hitl_override'`; later
  LLM passes will not overwrite it.
- **approve_label** — recorded as `hitl_approved`; the report can count
  human-verified labels.
- **merge_clusters** — documents reassigned to the target cluster; term lists
  and word frequencies recomputed from the merged membership.
- **split_cluster** — the macro cluster was decomposed into its micro-clusters
  (promoted to top level). On a flat run the same record flags the cluster in
  `needs_recluster` for targeted re-clustering instead.

The reviewed artifact is a normal run directory — the HITL app, the report
renderer and the future backend consume it exactly like an unreviewed one.